In [1]:
!pip install gradio transformers torch reportlab requests black autopep8

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 21.7 MB/s eta 0:00:00


In [2]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
import tempfile

def export_txt(content):
    temp = tempfile.NamedTemporaryFile(delete=False, suffix=".txt")
    with open(temp.name, "w") as f:
        f.write(content)
    return temp.name

def export_pdf(content):
    temp = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
    c = canvas.Canvas(temp.name, pagesize=letter)
    text = c.beginText(40, 750)

    for line in content.split("\n"):
        text.textLine(line)

    c.drawText(text)
    c.save()

    return temp.name

In [3]:
import ast

class DocGenieAnalyzer:

    @staticmethod
    def extract_function_signature(code):
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                params = []
                for arg in node.args.args:
                    param_type = ast.unparse(arg.annotation) if arg.annotation else "Any"
                    params.append({
                        "name": arg.arg,
                        "type": param_type,
                        "default": None
                    })

                return_type = ast.unparse(node.returns) if node.returns else "Any"

                return {
                    "name": node.name,
                    "params": params,
                    "return_type": return_type,
                    "node": node
                }

    @staticmethod
    def analyze_function_logic(func_node):
        has_loops = False
        has_conditions = False
        operations = set()
        function_calls = set()

        for node in ast.walk(func_node):
            if isinstance(node, (ast.For, ast.While)):
                has_loops = True

            if isinstance(node, ast.If):
                has_conditions = True

            if isinstance(node, ast.BinOp):
                if isinstance(node.op, ast.Add):
                    operations.add("addition")
                if isinstance(node.op, ast.Sub):
                    operations.add("subtraction")
                if isinstance(node.op, ast.Mult):
                    operations.add("multiplication")
                if isinstance(node.op, ast.Div):
                    operations.add("division")

            if isinstance(node, ast.Call):
                if hasattr(node.func, "id"):
                    function_calls.add(node.func.id)

        description = "Performs computation"
        if has_conditions:
            description += " with conditional logic"
        if has_loops:
            description += " and iterative processing"

        return {
            "has_loops": has_loops,
            "has_conditions": has_conditions,
            "operations": list(operations),
            "function_calls": list(function_calls),
            "description": description
        }

In [4]:
class DocstringGenerator:

    @staticmethod
    def generate_google_docstring(signature, analysis):
        lines = []
        lines.append(f'"""{signature["name"]} function.')
        lines.append("")
        lines.append(analysis["description"] + ".")
        lines.append("")
        lines.append("Args:")

        for param in signature["params"]:
            lines.append(f'    {param["name"]} ({param["type"]}): Parameter description.')

        lines.append("")
        lines.append("Returns:")
        lines.append(f'    {signature["return_type"]}: Return value description.')
        lines.append('"""')

        return "\n".join(lines)

In [7]:
# ====================== IMPORTS ======================
import ast
import gradio as gr
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
import tempfile

# ====================== CORE ANALYZER ======================
class DocGenieAnalyzer:

    @staticmethod
    def extract_function_signature(code):
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                params = []
                for arg in node.args.args:
                    param_type = ast.unparse(arg.annotation) if arg.annotation else "Any"
                    params.append({
                        "name": arg.arg,
                        "type": param_type,
                        "default": None
                    })

                return_type = ast.unparse(node.returns) if node.returns else "Any"

                return {
                    "name": node.name,
                    "params": params,
                    "return_type": return_type,
                    "node": node
                }
        return None

    @staticmethod
    def analyze_function_logic(func_node):
        has_loops = False
        has_conditions = False

        for node in ast.walk(func_node):
            if isinstance(node, (ast.For, ast.While)):
                has_loops = True
            if isinstance(node, ast.If):
                has_conditions = True

        description = "Performs computation"
        if has_conditions:
            description += " with conditional logic"
        if has_loops:
            description += " and iterative processing"

        return {
            "description": description
        }

    @staticmethod
    def generate_google_docstring(signature, analysis):
        lines = []
        lines.append(f'"""{signature["name"]} function.')
        lines.append("")
        lines.append(analysis["description"] + ".")
        lines.append("")
        lines.append("Args:")
        for param in signature["params"]:
            lines.append(f'    {param["name"]} ({param["type"]}): Parameter description.')
        lines.append("")
        lines.append("Returns:")
        lines.append(f'    {signature["return_type"]}: Return value description.')
        lines.append('"""')
        return "\n".join(lines)

    @staticmethod
    def generate_numpy_docstring(signature, analysis):
        lines = []
        lines.append(f'"""{signature["name"]} function.')
        lines.append("")
        lines.append(analysis["description"] + ".")
        lines.append("")
        lines.append("Parameters")
        lines.append("----------")
        for param in signature["params"]:
            lines.append(f'{param["name"]} : {param["type"]}')
            lines.append("    Parameter description.")
        lines.append("")
        lines.append("Returns")
        lines.append("-------")
        lines.append(f'{signature["return_type"]}')
        lines.append("    Return value description.")
        lines.append('"""')
        return "\n".join(lines)

# ====================== EXPORT FUNCTIONS ======================
def export_txt(content):
    temp = tempfile.NamedTemporaryFile(delete=False, suffix=".txt")
    with open(temp.name, "w") as f:
        f.write(content)
    return temp.name

def export_pdf(content):
    temp = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
    c = canvas.Canvas(temp.name, pagesize=letter)
    text = c.beginText(40, 750)

    for line in content.split("\n"):
        text.textLine(line)

    c.drawText(text)
    c.save()
    return temp.name

# ====================== GRADIO LOGIC ======================
generated_output = ""

def generate_docstring(code, style):
    global generated_output
    try:
        if not code.strip():
            return "⚠️ Please paste a Python function."

        sig = DocGenieAnalyzer.extract_function_signature(code)

        if sig is None:
            return "❌ No valid function detected. Please paste a proper Python function."

        logic = DocGenieAnalyzer.analyze_function_logic(sig["node"])

        if style == "google":
            doc = DocGenieAnalyzer.generate_google_docstring(sig, logic)
        else:
            doc = DocGenieAnalyzer.generate_numpy_docstring(sig, logic)

        generated_output = doc + "\n\n" + code
        return generated_output

    except Exception as e:
        return f"🚨 Internal Error:\n{str(e)}"

def download_txt():
    return export_txt(generated_output)

def download_pdf():
    return export_pdf(generated_output)

# ====================== GRADIO UI ======================
with gr.Blocks() as demo:
    gr.Markdown("# 🧞 Doc-Genie – Professional Python Docstring Generator")

    with gr.Row():
        with gr.Column():
            code_input = gr.Textbox(lines=15, label="Paste Python Function Code")
            style = gr.Radio(["google", "numpy"], value="google", label="Docstring Style")
            btn = gr.Button("Generate Docstring")

        with gr.Column():
            output = gr.Textbox(lines=20, label="Generated Output")

    btn.click(generate_docstring, inputs=[code_input, style], outputs=output)

    with gr.Row():
        txt_btn = gr.Button("📄 Download TXT")
        pdf_btn = gr.Button("📕 Download PDF")

    txt_btn.click(download_txt, outputs=gr.File())
    pdf_btn.click(download_pdf, outputs=gr.File())

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://55d27f3a62c547a2cb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
import tempfile
import os

def export_txt(content):
    temp = tempfile.NamedTemporaryFile(delete=False, suffix=".txt")
    with open(temp.name, "w") as f:
        f.write(content)
    return temp.name

def export_pdf(content):
    temp = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
    c = canvas.Canvas(temp.name, pagesize=letter)
    text = c.beginText(40, 750)

    for line in content.split("\n"):
        text.textLine(line)

    c.drawText(text)
    c.save()

    return temp.name